In [1]:
#-------------------------------------------------------------------------------
# Import modules
#-------------------------------------------------------------------------------
import torch
import pandas as pd
import numpy as np
import lightning as pl
import glob
import os
import ollama
import transformers
import re
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_lightning import seed_everything
def is_notebook():return True if 'JPY_PARENT_PID' in os.environ else False
USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
current_directory = os.getcwd()
seed_everything(42)

#-------------------------------------------------------------------------------
## Initial Ollama Installation ##
#-------------------------------------------------------------------------------
#!python -m venv ollama
#!pip install ollama
# Models pulled = llama3, mistral, llama2, vicuna

#-------------------------------------------------------------------------------
# Import value profiles
#-------------------------------------------------------------------------------
value_questions = pd.read_csv('ValueQuestions.csv', low_memory=False)
original_answers = pd.read_csv('OriginalValueAnswers.csv', low_memory=False)
edited_answers = pd.read_csv('EditedValueAnswers.csv', low_memory=False)
questions = value_questions['Questions'][1:35].tolist()
no_questions = len(questions)

### NEED TO SPLIT INTO RANKINGS OUT OF 10 AND OUT OF 4

#-------------------------------------------------------------------------------
# Scenario lists assigned to dyads
#-------------------------------------------------------------------------------
no_dyads = 15
scenario_list = [[] for _ in range(no_dyads)]
scenario_list[0] = ['scenario_2_4', 'scenario_3_5', 'scenario_6_8']
scenario_list[1] = ['scenario_3_5', 'scenario_5_7', 'scenario_7_24']
scenario_list[2] = ['scenario_2_4', 'scenario_5_7', 'scenario_8_36']
scenario_list[3] = ['scenario_2_4', 'scenario_4_31', 'scenario_7_24']
scenario_list[4] = ['scenario_2_4', 'scenario_4_31', 'scenario_7_24']
scenario_list[5] = ['scenario_2_4', 'scenario_3_5', 'scenario_7_24']
scenario_list[6] = ['scenario_2_4', 'scenario_3_5', 'scenario_7_24']
scenario_list[7] = ['scenario_3_5', 'scenario_6_8', 'scenario_8_36']
scenario_list[8] = ['scenario_3_5', 'scenario_6_8', 'scenario_8_36']
scenario_list[9] = ['scenario_1_2', 'scenario_3_5', 'scenario_8_36']
scenario_list[10] = ['scenario_2_4', 'scenario_4_31', 'scenario_6_8']
scenario_list[11] = ['scenario_1_2', 'scenario_3_5', 'scenario_4_31']
scenario_list[12] = ['scenario_1_2', 'scenario_4_31', 'scenario_7_24']
scenario_list[13] = ['scenario_6_8', 'scenario_3_5', 'scenario_2_4']
scenario_list[14] = ['scenario_2_4', 'scenario_5_7', 'scenario_7_24']
print('Number of Dyads:', no_dyads)

/blue/prismap-ai-core/vnolan/conda/envs/APARI_Models/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[rank: 0] Seed set to 42


Number of Dyads: 15


In [2]:
#---------------------------------------------------------------------
# SCENARIOS AND ANSWERS
#---------------------------------------------------------------------
answer_subject = '''
Answer 1 =  Do not perform CPR if it does not improve a chance at full or partial recovery.
Answer 2 =	Do not use a ventilator if it does not allow a chance at full or partial recovery.
Answer 3 =	Kept comfortable and pain free while nature takes its course.
Answer 4 =	Everything to be done even if it looks like there is no chance of full or partial recovery.
Answer 5 =	Proxy to make this decision on his or her own. '''

answer_proxy = '''
Answer 1 =	Do not perform CPR if it does not improve a chance at full or partial recovery.
Answer 2 =	Do not use a ventilator if it does not allow a chance at full or partial recovery.
Answer 3 =	Kept comfortable and pain free while nature takes its course.
Answer 4 =	Everything to be done even if it looks like there is no chance of full or partial recovery.
Answer 5 =	Proxy to make this decision on his or her own. '''

scenarios = {
'scenario_1_2': ''' Age: 43 
Background: You enjoy an active lifestyle. You like marathons, hiking, and rock climbing. You have been married for 20 years and have two children (16 years and 12 years). You have occasional seasonal allergies, but otherwise, you have been in good health.
Living Situation: You live in a two-story home with a large backyard with a garden. Together with your spouse, you take care of the children, managing their daily activities, schoolwork, and extracurricular activities. 
Event: On one of your rock-climbing adventures at the gym, you fell from a significant height although you were using a safety harness. The angle and force of the fall caused you to hit your head hard against the climbing wall and you lost consciousness. 
You were examined by medical personnel on site at the gym who transported you to the hospital. The ER diagnosed you with a traumatic brain injury and you were unable to communicate or make decisions for yourself for some time. However, you did get better over the next year. 
On one occasion when you were taking a shower, you noticed a lump in your upper left thigh. The lump seemed to be getting bigger over time, so you decided to talk with your doctor. Out of precaution, the doctor took a biopsy.
Investigations: The biopsy revealed a high-grade soft tissue sarcoma (a malignant cancer). An MRI of the leg and pelvis was done, and it confirmed the cancer had spread to the nearby structures.
Clinical Decision: Because the cancer had spread, treatment had to be aggressive. The best chance for a cure would be to remove part of your hip bone [hemipelvectomy] and amputate your left leg at the hip. ''', 

'scenario_2_4': ''' Age: 20
Background: You are a university student studying literature. You are an introvert and struggle with severe depression, but you have sought counseling through the university’s mental health services. You are an only child, and your parents live in another city, about a 3-hour drive away.
Living Situation: You live alone in a single room in the university dormitory and have a few close friends, but mostly you keep to yourself. Your grades are great, and you are passionate about poetry and literature.
Event: Emergency services were called to the university dormitory after reports of a fire. Upon arrival, they found you with extensive burn injuries. Initial investigation suggested you tried to set yourself on fire in your room.
Investigations: You were taken to the nearest burn center, and they confirmed that you had third-degree burns covering approximately 80% of your total body, including your face, chest, arms, and legs. Due to the severity of the burns and a potential airway problem, you were placed on a breathing machine (intubated) and sedated.
Clinical Decision: The burn and surgical teams were faced with the challenge of how to care for you because you have a high risk of infection, complications, and death. They decided to keep you stable, making sure you have adequate fluid intake and that they prevent you from getting an infection. 
Long-term, you will require multiple surgeries, skin grafts (replacing burnt skin with good skin), and extensive rehab. Since you had prior mental health challenges and your injuries were severe, they insisted on a psychiatric evaluation and support.''',

'scenario_3_5': ''' Age: 70 
Background: You are a retired librarian who has lived a rich and fulfilling life. You have two adult children and four grandchildren. You recently had vascular surgery because of peripheral vascular disease (disease of the blood vessel). 
Three days after your surgery and while in the hospital, you had multiple conversations about how content you were with your life and would like nature run its course rather than pursue aggressive medical treatment.
Living Situation: You live alone in a small apartment after losing your spouse five years ago. Your children visit frequently, and you enjoy spending time reading and gardening.
Event: A week after your operation, you started to have symptoms of respiratory distress. You developed a fever, persistent cough, and had difficulty breathing. Because of that, you were placed on a ventilator and the team put you to sleep (sedation), but you still did not improve. 
Investigations: They frequently checked your oxygen saturation level, but it kept on decreasing rapidly. So, they did a chest X-ray, and it was confirmed that you had severe COVID-19 pneumonia.
Clinical Decision: The medical team decided to consider an advanced treatment due to the severity of the pneumonia and your decreasing oxygen levels. They decided on Extracorporeal membrane oxygenation (ECMO) as it could potentially support your lungs and give you a chance of recovery. 
However, the hospital has limited ECMO machines, and there are other critically ill COVID-19 patients who might also benefit from use of this machine. ''',

'scenario_4_31': ''' Age: 35 
Background: You are an accomplished architect, and your work was featured in various architectural magazines. Aside from your professional achievements, you were an active community volunteer, often leading urban gardening projects and neighborhood clean-ups. 
Living Situation: You live in a penthouse of a building you designed, with a rooftop garden. 
Event: While attending a construction site for a new project, an unexpected structural collapse occurred, and you were trapped under heavy debris for several hours before rescue teams could safely extract you. You were immediately transported to the trauma center of a nearby hospital with multiple injuries. 
Investigations: On evaluation, they diagnosed you with severe traumatic brain injury (TBI), multiple rib fractures, a punctured lung, and a crushed pelvis. 
Clinical Decision: They placed you on a ventilator due to respiratory distress and performed emergency surgery to stabilize your injuries. After the surgery, they kept you in a coma to help minimize your brain injury. The medical team stated that there was a high chance you will have long-term mental impairment if you did recover, or you will remain in coma state.''',

'scenario_5_7': ''' Age: 77 
Background: You are a widow(er) and retired from a career in accounting. You have a history of multiple heart diseases and diabetes, but you have never smoked. However, your spouse smoked for many years and exposed you to second-hand smoke during that time.
Living Situation: You live alone in a one-bedroom apartment and have two adult children who live nearby and visit frequently. They assist you with grocery shopping, house chores, and take you to doctor’s appointment.
Event: One day, you were admitted to the hospital with severe breathlessness and fatigue. The initial evaluation suggested severe heart failure.
Investigations: The medical team find out that your heart condition is very poor, and you were expected to survive for just a few months. During your hospitalization, they ordered a routine chest Xray which showed a small mass in your right lung. 
Clinical Decision: The standard treatment for a mass is biopsy. If the biopsy is positive for cancer, then, the next step would be removal of the mass. However, given your severe heart failure and your survival diagnosis, the surgery would pose a high risk, even death. ''',

'scenario_6_8': ''' Age: 69  
Background: You are a retired schoolteacher and widowed about a decade ago. You have two grown-up children and three grandchildren, and you were diagnosed with chronic obstructive pulmonary disease (COPD). He had spent many years of smoking, but quit 5 years ago. 
Living Situation: You live in a senior community where you have many friends. Your daughter lives nearby and visits often, helping you with errands and doctor's appointments. 
Event: You were admitted to the hospital after a minor fall at your home. Though you didn't get any fractures, your oxygen levels were unstable, meaning you had a COPD exacerbation episode. 
Investigations: During your hospital stay, your breathing difficulties got worse and then you started to have trouble swallowing, especially coughing during meals. A swallow evaluation showed muscle weakness from the result of having COPD for a long time. 
Clinical Decision: The standard approach for difficulty in swallowing is to consider inserting a tube in the stomach [percutaneous endoscopic gastrostomy (PEG)] for adequate nutrition. However, given your overall health status and your advanced COPD, the team felt a conservative management with dietary modifications might be more appropriate. ''',

'scenario_7_24': ''' Age: 68 years 
Background: You are a former coal miner who was diagnosed with chronic obstructive pulmonary disease (COPD) fifteen years ago. Over the years, your health has progressively declined. You have been on home oxygen therapy for the past five years and have had multiple hospital admissions due to exacerbations of your COPD. 
Living Situation: You are widowed and have lived alone since your spouse passed away three years ago. Since then, your adult son has taken an active role in your care, assisting you with medical appointments and managing your medications. 
Event: You were brought to the emergency department with a severe shortness of breath following a minor respiratory infection. 
Investigations: You were placed on a ventilator when examination showed that you had respiratory failure. 
Clinical Decision: Despite treatments, you did not show significant improvement. In addition, you were on a ventilator for a long time and the doctors were concerned about ventilator-associated complications. The team assessed that you might do better with a tracheostomy (a hole in your throat) which will be necessary for you to breathe better for the rest of your life. ''',

'scenario_8_36': ''' Age: 78 
Background: You are a former university professor that has been living with advanced Parkinson's disease for over a decade. You have issues with moving around and frequent falls. You also have a history of hypertension and chronic kidney disease. 
Despite these health challenges, you always maintained a positive outlook. During one of your outpatient visits, you mentioned that you wanted all possible treatments, even if the chances of recovery were slim. You always expressed a wish to fight every battle, no matter the odds.
Living Situation: You lived alone but had a regular caretaker that came to help you daily.
Event: You were admitted to the hospital following a significant fall at home, resulting in a hip fracture. During the hospital stay, you developed a hospital-acquired infection which led to an infection spreading to the blood. 
Clinical Decision: The infection in the blood combined with other conditions resulted in a rapid decline in your health. You became unstable and required blood pressure support. The medical team provided you with a strong antibiotic and gave you IV fluid regularly, but improvement in your condition was uncertain. ''' }

In [3]:
#------------------------------------------------------------------------
# LOOPING OVER DYADS AND SCENARIOS
#------------------------------------------------------------------------
subject_scenario1 = [5,3,3,3,5,3,2,3,3,3,1,4,3,2,4]
subject_scenario2 = [3,3,4,3,3,3,4,4,3,3,1,3,3,3,3]
subject_scenario3 = [5,3,4,4,3,3,3,3,3,3,3,3,3,3,4]
proxy_avg_results = [2,1,2,2,1,3,0,0,0,2,0,3,2,0,1]
temperature_values = [0.0,0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
no_temps = len(temperature_values)
no_trials = 10
columns = ['Dyad', 'Temperature', 'Trial', 'AI Subject Without Values', 'AI Subject With Values','AI Proxy Without Values', 'AI Proxy With Values', 'Human Proxy']

# Load existing data from CSV
try:
    results_df = pd.read_csv('OllamaTrials2.csv')
    processed_dyads = results_df['Dyad'].unique()
    processed_dyads = set(processed_dyads)  # Using set for faster membership checking
except FileNotFoundError:
    results_df = pd.DataFrame(columns=columns)
    processed_dyads = set()

start_dyad = 0 
start_temp = 0.0
start_trial = 0
skip_dyads = True
skip_temps = True
skip_trials = True

## FUNCTION TO EXTRACT THE FIRST NUMBER FROM THE TEXT
def extract_first_number(text):
    match = re.search(r'\d+', text)
    if match: return match.group()
    return None

## LOOPING OVER ALL COMBINATIONS OF PARAMETERS
for dyad in range(no_dyads):
    if skip_dyads:
        if dyad < start_dyad: 
            continue
        else: 
            skip_dyads = False
    print(f'############################### DYAD {dyad+1} #####################################')

    ### OBTAINING VALUES FOR SUBJECT
    values_subject = original_answers.iloc[dyad][2:36]
    updated_values_subject = [f"{question} Answer = {value}. " for question, value in zip(questions, values_subject)]
    updated_values_subject = ''.join(updated_values_subject)

    ### OBTAINING ANSWERS FOR SUBJECT
    subject_answers = [subject_scenario1[dyad], subject_scenario2[dyad], subject_scenario3[dyad]]  

    for temp in range(no_temps):
        temperature = temperature_values[temp]
        if skip_temps:
            if dyad == start_dyad and temperature < start_temp:
                continue
            else:
                skip_temps = False
        print(f'Temperature is {temperature_values[temp]}')

        for trial in range(no_trials):
            if skip_trials:
                if dyad == start_dyad and temperature == start_temp and trial < start_trial:
                    continue
                else:
                    skip_trials = False
            print(f'************* Trial {trial +1} ****************')  
            SUBJECT_NOVALS = []
            SUBJECT_VALS = []
            PROXY_NOVALS = []
            PROXY_VALS = []

            for index, scenario_code in enumerate(scenario_list[dyad]):      

                ### PART 1: SUBJECT ANSWERING FOR THEMSELVES        
                prompt_text1 = '''
                ### TASK: Imagine that you are the patient in this scenario: {SCENARIO_TEXT}. 
                A major clinical decision needs to be made for your treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the patient, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario. 
                Give your answer as a single number corresponding to one of the five options.'''        
                scenario_text = scenarios[scenario_code]
                prompt1 = prompt_text1.format(SCENARIO_TEXT=scenario_text, ANSWERS=answer_subject)
                response1 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt1}])  
                SUBJECT_NOVALS.append(extract_first_number(response1['message']['content']))

                prompt_text2 = '''         
                ### TASK: Imagine that you are the patient in this scenario: {SCENARIO_TEXT}. 
                The patient (you) were first asked to fill out two questionnaires about your values and wishes in life. These are your answers: {VALUES}. 
                A major clinical decision needs to be made for your treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the patient, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario and the questions from the values questionnaires.        
                Give your answer as a single number corresponding to one of the five options.'''        
                scenario_text = scenarios[scenario_code]
                prompt2 = prompt_text2.format(SCENARIO_TEXT=scenario_text, ANSWERS=answer_subject, VALUES = updated_values_subject)
                response2 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt2}])  
                SUBJECT_VALS.append(extract_first_number(response2['message']['content']))

                ### PART 2: PROXY ANSWERING FOR SUBJECT   
                prompt_text3 = '''
                ### TASK: Imagine that you are the decision-making proxy (e.g. family member or friend) of the patient in this scenario: {SCENARIO_TEXT}. 
                A major clinical decision needs to be made by you for the patient's treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the proxy, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario. 
                Give your answer as a single number corresponding to one of the five options.'''          
                scenario_text = scenarios[scenario_code]
                prompt3 = prompt_text3.format(SCENARIO_TEXT=scenario_text, ANSWERS=answer_proxy)
                response3 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt3}])
                PROXY_NOVALS.append(extract_first_number(response3['message']['content']))

                prompt_text4 = '''         
                ### TASK: Imagine that you are the decision-making proxy (e.g. family member or friend) of the patient in this scenario: {SCENARIO_TEXT}. 
                The patient was first asked to fill out two questionnaires about their values and wishes in life. These are their answers: {VALUES}. 
                A major clinical decision needs to be made by you for the patient's treatment in this scenario, and there are five possible choices that you can choose from: {ANSWERS}.
                Imagining that you are the proxy, choose only one of the answers that you think would be most appropriate to be applied in this situation using the information in the scenario and the questions from the values questionnaires. 
                Give your answer as a single number corresponding to one of the five options.'''   
                scenario_text = scenarios[scenario_code]
                prompt4 = prompt_text4.format(SCENARIO_TEXT=scenario_text, ANSWERS=answer_subject, VALUES = updated_values_subject)
                response4 = ollama.chat(model='llama3', options={'temperature': temperature_values[temp]}, messages=[{'role': 'system', 'content': prompt4}])
                PROXY_VALS.append(extract_first_number(response4['message']['content']))
                
            def count_exact_matches(subject_answers, patient_answers):
                count = 0
                for subj_ans, pat_ans in zip(subject_answers, patient_answers):
                    if str(subj_ans) == pat_ans: count += 1
                return count
            SUBJECT_NOVALS_CORRECT = count_exact_matches(subject_answers, SUBJECT_NOVALS)
            SUBJECT_VALS_CORRECT = count_exact_matches(subject_answers, SUBJECT_VALS)
            PROXY_NOVALS_CORRECT = count_exact_matches(subject_answers, PROXY_NOVALS)
            PROXY_VALS_CORRECT = count_exact_matches(subject_answers, PROXY_VALS)
 
            new_row = pd.DataFrame([{
                'Dyad': dyad + 1,
                'Temperature': temperature_values[temp],
                'Trial': trial + 1,
                'AI Subject Without Values': SUBJECT_NOVALS_CORRECT,
                'AI Subject With Values': SUBJECT_VALS_CORRECT,
                'AI Proxy Without Values': PROXY_NOVALS_CORRECT,
                'AI Proxy With Values': PROXY_VALS_CORRECT,
                'Human Proxy': proxy_avg_results[dyad]
            }])

            results_df = pd.concat([results_df, new_row], ignore_index=True)
            results_df.to_csv('OllamaTrials2.csv', index=False)


############################### DYAD 1 #####################################
Temperature is 0.0
************* Trial 1 ****************


/scratch/local/5365552/ipykernel_3422903/3820246275.py:140: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, new_row], ignore_index=True)


************* Trial 2 ****************
************* Trial 3 ****************
************* Trial 4 ****************
************* Trial 5 ****************
************* Trial 6 ****************
************* Trial 7 ****************
************* Trial 8 ****************
************* Trial 9 ****************
************* Trial 10 ****************
Temperature is 0.2
************* Trial 1 ****************
************* Trial 2 ****************
************* Trial 3 ****************
************* Trial 4 ****************
************* Trial 5 ****************
************* Trial 6 ****************
************* Trial 7 ****************
************* Trial 8 ****************
************* Trial 9 ****************
************* Trial 10 ****************
Temperature is 0.4
************* Trial 1 ****************
************* Trial 2 ****************
************* Trial 3 ****************
************* Trial 4 ****************
************* Trial 5 ****************
************* Trial 6 **